# Section 7 — Dispatch Failure & Lakebase Memory
### source: MULTI_AGENT/dispatch_failure_2.ipynb

**Exam objective:**
> *Given a multi-agent system where a coordinating agent has decomposed a task and dispatched sub-agents, diagnose the root cause (sub-agents receiving individual task messages rather than full agent traces at dispatch time), and select the configuration that resolves the failure without expanding each sub-agent's context window.*


**What this notebook illustrates:**

A Python orchestrator (no Databricks Supervisor Agent endpoint) answers:
> "What are the top product categories by revenue for our best customers?"

At turn 3 the orchestrator establishes four scoping decisions and writes them to **Lakebase long-term memory**.

We compare two dispatch strategies to the Genie sub-agent (`customer_genie`):
- **dispatch_A** — task message only. Genie starts fresh, decisions unknown.
- **dispatch_B** — task + decisions retrieved from Lakebase **just before** dispatch.


**Genie space tables** (`demo.demo.*` from the `customer_genie` demo):
- `customers` — customer_id, first_name, last_name, email, registration_date, **tier**
- `orders` — order_id, customer_id, order_date, total_amount, channel, **status**
- `order_items` — order_id, product_id, quantity, **unit_price**, **discount_amount**
- `products` — product_id, product_name, **category**, cost_price
- `shipping_zones` — zone_id, region, shipping_cost


## 1 — Setup

In [ ]:
# Install a most recent version of databricks sdk with the postgres package
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "databricks-sdk>=0.81.0"])

# Then restart kernel 

Load the environment variables

In [ ]:
import os, json, time, uuid, re, textwrap
from dotenv import load_dotenv

load_dotenv()

assert os.getenv('DATABRICKS_HOST'),    'DATABRICKS_HOST not set'
assert os.getenv('DATABRICKS_TOKEN'),   'DATABRICKS_TOKEN not set'
assert os.getenv('GENIE_SPACE_ID'),     'GENIE_SPACE_ID not set  (copy from your Genie space URL)'
assert os.getenv('MLFLOW_TRACKING_URI'),'MLFLOW_TRACKING_URI not set'


In [ ]:

HOST              = os.getenv('DATABRICKS_HOST').rstrip('/')
TOKEN             = os.getenv('DATABRICKS_TOKEN')
GENIE_SPACE_ID    = os.getenv('GENIE_SPACE_ID')
SESSION_ID        = str(uuid.uuid4())
SUFFIX            = SESSION_ID[:6]

PGDATABASE="databricks_postgres"
# TODO: replace with your databricks account email
PGUSER="oliver@mlops-media.com"
# TODO: replace with the name of your own Lakebase endpoint
project_id = "demo-lake-memory"
branch_id = "development"
endpoint_id = "primary"
LAKEBASE_ENDPOINT_NAME = f'projects/{project_id}/branches/{branch_id}/endpoints/{endpoint_id}'

print(f'host={HOST[:11]}{HOST[25:]}')
print(f'genie_space={GENIE_SPACE_ID[:3]}')
print(f'session_id={SESSION_ID}')

In [ ]:
import mlflow, tiktoken, requests
from sqlalchemy import create_engine, text
from databricks.sdk import WorkspaceClient

w   = WorkspaceClient()

enc = tiktoken.get_encoding('cl100k_base')
def count_tokens(t): return len(enc.encode(t or ''))

mlflow.set_experiment(f'/Users/{PGUSER}/s7_dispatch_lakebase')
print('Ready.')

In [ ]:
# Get project details
project = w.postgres.get_project(name=f'projects/{project_id}')

print(f"Project: {project.name}")
print(f"Display name: {project.status.display_name}")
print(f"Postgres version: {project.status.pg_version}")

## 2 — Lakebase connection

OAuth token generated by the SDK — no hardcoded password.

Docs: https://docs.databricks.com/aws/en/oltp/instances/query/notebook

In [ ]:
import psycopg

def get_conn():
    """Fresh OAuth token → fresh connection. Token valid 1h."""
    cred = w.postgres.generate_database_credential(endpoint=LAKEBASE_ENDPOINT_NAME)
    endpoint = w.postgres.get_endpoint(name=LAKEBASE_ENDPOINT_NAME)
    return psycopg.connect(
        host=endpoint.status.hosts.host, 
        dbname=PGDATABASE,
        user=PGUSER, 
        password=cred.token,
        sslmode="require"
    )

# DDL — une seule fois
with get_conn() as conn:
    conn.execute("""
        CREATE TABLE IF NOT EXISTS public.multiagent_session_memory (
            session_id     TEXT NOT NULL,
            turn           INT  NOT NULL,
            decision_key   TEXT NOT NULL,
            decision_value TEXT NOT NULL,
            PRIMARY KEY (session_id, decision_key)
        )
    """)
    conn.commit()
    print("Table ready.")

## 4 — Business scenario

User question + four decisions aligned at turn 3.
All decisions reference the actual `customer_genie` table columns.

In [ ]:
USER_QUESTION = "What are the top product categories by revenue for our best customers?"

# Decisions established at turn 3 — written to Lakebase now
# Grounded in actual customer_genie table columns
SESSION_DECISIONS = [
    {"turn": 3, "decision_key": "tier_filter",
     "decision_value": "Hard filter: WHERE customers.tier = 'Gold'. "
                       "'Best customers' is DEFINED as tier = 'Gold' — do NOT rank by lifetime_value."},
    {"turn": 3, "decision_key": "status_filter",
     "decision_value": "Hard filter: WHERE orders.status = 'delivered'. Exclude every other status."},
]

GENIE_TASK = "What are the top product categories by revenue for our best customers?"


## 5 — Write decisions to Lakebase at turn 3

Decisions are written when established — not at dispatch time.
This is the **long-term memory** pattern for agents.

In [ ]:
def mem_write(session_id, decisions):
    with get_conn() as conn:
        for d in decisions:
            conn.execute("""
                INSERT INTO public.multiagent_session_memory
                    (session_id, turn, decision_key, decision_value)
                VALUES (%s, %s, %s, %s)
                ON CONFLICT (session_id, decision_key)
                DO UPDATE SET decision_value = EXCLUDED.decision_value
            """, (session_id, d["turn"], d["decision_key"],
                  d["decision_value"]))
        conn.commit()
    print(f"Written {len(decisions)} decisions.")

def mem_read(session_id):
    with get_conn() as conn:
        rows = conn.execute(
            "SELECT decision_key, decision_value FROM public.multiagent_session_memory "
            "WHERE session_id = %s ORDER BY turn",
            (session_id,)
        ).fetchall()
    return [{"decision_key": r[0], "decision_value": r[1]} for r in rows]


In [ ]:
mem_write(SESSION_ID, SESSION_DECISIONS)
rows = mem_read(SESSION_ID)
for r in rows:
    print(f"  {r['decision_key']:<22}: {r['decision_value'][:60]}")

## 6 — Genie sub-agent wrapper

Each call starts a **fresh Genie conversation**.
Genie has no memory of the orchestrator session — only the dispatch message.

In [ ]:
def genie_query(prompt, max_wait=90):
    headers = {'Authorization': f'Bearer {TOKEN}', 'Content-Type': 'application/json'}
    base    = f'{HOST}/api/2.0/genie/spaces/{GENIE_SPACE_ID}'

    r    = requests.post(f'{base}/start-conversation', headers=headers, json={'content': prompt})
    r.raise_for_status()
    data = r.json()
    cid, mid = data['conversation_id'], data['message_id']

    for _ in range(max_wait // 2):
        r   = requests.get(f'{base}/conversations/{cid}/messages/{mid}', headers=headers)
        r.raise_for_status()
        msg = r.json()
        if msg.get('status') == 'COMPLETED':
            sql, txt = '', ''
            for att in msg.get('attachments', []):
                if 'query' in att:
                    sql = att['query'].get('query', '')
                if 'text' in att:
                    txt = att['text'].get('content', '')
            return {'sql': sql, 'result_text': txt, 'status': 'ok'}
        if msg.get('status') in ('FAILED', 'CANCELLED'):
            return {'sql': '', 'result_text': '', 'status': msg.get('status')}
        time.sleep(2)
    return {'sql': '', 'result_text': '', 'status': 'TIMEOUT'}

print('Genie wrapper ready.')

## 7 — Candidate pool: two dispatch strategies

1- dispatch_A : No lakebase
2- dispatch_B : lakebase read just before dispatch.

In [ ]:
def build_dispatch_A():
    """Anti-pattern: task message only — no Lakebase read."""
    return GENIE_TASK

def build_dispatch_B(session_id):
    """Correct: read decisions from Lakebase just before dispatch."""
    with get_conn() as conn:
        rows = conn.execute(
            'SELECT decision_key, decision_value FROM multiagent_session_memory '
            'WHERE session_id = %s ORDER BY turn',
            (session_id,)
        ).fetchall()
    if not rows:
        return GENIE_TASK
    block = '\n'.join(f"- {r[0].replace('_',' ').title()}: {r[1]}" for r in rows)
    return textwrap.dedent(f'''
        Task: {GENIE_TASK}

        Apply all of the following constraints (retrieved from session memory):
        {block}

        Return: product category, net revenue, sorted descending.
    ''').strip()


In [ ]:
prompt_A = build_dispatch_A()
prompt_B = build_dispatch_B(SESSION_ID)

CANDIDATES = [
    {'id': 'dispatch_A', 'label': 'Task only — no Lakebase',   'prompt': prompt_A},
    {'id': 'dispatch_B', 'label': 'Task + Lakebase decisions',  'prompt': prompt_B},
]

print(f"{'ID':<14} {'Tokens':>7}  Label")
print('-'*50)
for c in CANDIDATES:
    print(f"{c['id']:<14} {count_tokens(c['prompt']):>7}  {c['label']}")

print('\n── dispatch_B prompt preview ──')
print(prompt_B)

## 8 — Eval checks

Two discriminating checks aligned with the actual SESSION_DECISIONS.
We score the **generated SQL** — not the prompt.


In [ ]:
def check_gold_tier(sql):
    """Q1 — Does the SQL filter customers.tier = 'Gold'?"""
    u = sql.upper().replace('`', '')          
    joins_customers = 'CUSTOMERS' in u
    filters_gold    = bool(re.search(r"TIER\s*=\s*['\"]?GOLD", u, re.I))
    return round((0.4 if joins_customers else 0) + (0.6 if filters_gold else 0), 2)

def check_delivered_status(sql):
    """Q2 — Does the SQL filter orders.status = 'delivered'?"""
    u = sql.upper().replace('`', '')         
    has_status    = 'STATUS' in u
    has_delivered = bool(re.search(r"STATUS\s*=\s*['\"]?DELIVERED", u, re.I))
    return round((0.3 if has_status else 0) + (0.7 if has_delivered else 0), 2)


In [ ]:
EVAL_CHECKS = [
    {'id': 'Q1', 'name': 'Gold tier  (customers.tier)',    'fn': check_gold_tier,       'w': 0.5},
    {'id': 'Q2', 'name': 'Delivered  (orders.status)',     'fn': check_delivered_status, 'w': 0.5},
]

def score_sql(sql):
    s   = {c['id']: c['fn'](sql) for c in EVAL_CHECKS}
    avg = round(sum(c['w'] * s[c['id']] for c in EVAL_CHECKS), 3)
    return {**s, 'avg': avg}

print('Eval checks ready:')
for c in EVAL_CHECKS:
    print(f"  {c['id']}  {c['name']}  (w={c['w']})")

## 9 — Manual grid

Expected:
- **dispatch_A** → no tier/status filter → Q1=0, Q2=0
- **dispatch_B** → Gold + delivered → Q1=1, Q2=1


In [ ]:
def eval_dispatch(c):
    tok  = count_tokens(c['prompt'])
    out  = genie_query(c['prompt'])
    sql  = out.get('sql', '')
    s    = score_sql(sql)
    mem  = 'yes' if c['id'] == 'dispatch_B' else 'no'
    return {**c, 'sql': sql, 'scores': s, 'tokens': tok}

results = [eval_dispatch(c) for c in CANDIDATES]


In [ ]:
# Dispatch A:
dispatch = results[0]
print(f"{dispatch['id']}, {dispatch['label']}, tokens : {dispatch['tokens']}, scores {dispatch['scores']}")
print("SQL : ")
print(dispatch["sql"])

- Best customers are defined by lifetime value.
- No Filter on 'delivered' status !

In [ ]:
# Dispatch B:
dispatch = results[1]
print(f"{dispatch['id']}, {dispatch['label']}, tokens : {dispatch['tokens']}, scores {dispatch['scores']}")
print("SQL : ")
print(dispatch["sql"])

Filtered :

- customers_feature.tier = 'Gold'
- AND orders_feature.status = 'delivered'

As required

Perfect scores !

## 10 — Takeaways


**The fix**: decisions exit the conversation context → written to Lakebase at turn 3
→ retrieved on demand before dispatch → minimum context, complete signal.

Sources: [Lakebase](https://docs.databricks.com/aws/en/oltp/instances/query/notebook) · 
[Genie API](https://docs.databricks.com/aws/en/genie/genie-as-agent-tool) · 
[Multi-agent](https://docs.databricks.com/aws/en/generative-ai/agent-framework/multi-agent-apps)
